# Land use from LCS Meta-analysis 

THINGS TO DO:
- Land use from m2/FU to m2 --> need to multiply by production, where is the produciton in the spreadsheet?
    - Multiply land use values for arable (animal) products by the total UK production. This should result in a total land area, likely to be different from the total arable (pasture) area in the UK
- Scale all the land use factors associated with arable and animal products so the total arable and pasture land matches the totals in the UK

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../')))

In [3]:
# read csv file skipping first two lines
dataset_csv = pd.read_csv('data/LCA+Meta-Analysis+of+Food+Products+-+Full+Model+v0+(1) - Database.csv', skiprows=2, dtype=str)

In [4]:
european_countries = [
    'Austria',
    'Belgium',
    'Czech Republic',
    'Denmark',
    'Finland',
    'France',
    'Germany',
    'Ireland',
    'Italy',
    'Netherlands',
    'Norway',
    'Poland',
    'Portugal',
    'Romania',
    'Russian Federation',
    'Spain',
    'Sweden',
    'Switzerland',
    'United Kingdom']
# remove the countries that are not in the list
dataset_csv = dataset_csv[dataset_csv['Country'].isin(european_countries)]

In [5]:
# split into two datasets, one for animal and one for plant products
animal_product_types = [
    "Beef",
    "Brook trout, brown trout, rainbow trout, arctic char",
    "Chicken",
    "Chicken's eggs",
    "Common carp, tench, roach, perch, pike",
    "Common carp, tench, roach, perch, sander, pike",
    "Cow's cheese",
    "Cow's cheese (Emmental)",
    "Cow's cheese (Gouda)",
    "Cow's milk",
    "Duck",
    "Goose (foie gras)",
    "Lamb",
    "Lamb and mutton",
    "Pork",
    "Rainbow trout",
    "Salmon",
    "Sea bass, sea bream",
    "Sheep's milk",
    "Trout",
    "Trout (large)",
    "Trout (small)",
    "Turkey"]

dataset_csv_animal = dataset_csv[dataset_csv['Product'].isin(animal_product_types)]
# plant items = not animal items
dataset_csv_plant = dataset_csv[~dataset_csv['Product'].isin(animal_product_types)]

In [6]:
# print len full dataset, animal dataset and plant dataset (check)
print(f"Full dataset: {len(dataset_csv)}")
print(f"Animal dataset: {len(dataset_csv_animal)}")
print(f"Plant dataset: {len(dataset_csv_plant)}")
dataset_csv_plant.head()

Full dataset: 1155
Animal dataset: 390
Plant dataset: 765


,Unnamed: 0,Reference,Product,Scientific Name,Sys,Production Sys Details,Met,Methd Details,Country,Geog. Specific,...,Steel (kg),Alum (kg),Iron (kg),Concrt (kg),Wood (kg),Conv 1 (Conv 1 Wt MW-1 or LW-1),Econ Alloc (%),Conv 2 (RW Conv1 Wt-1),Water Use (L freshwater / kg or L final weight),Notes (Inventory)
7,NaN,Moudrý Jr et al. (2013b),Wheat bread,Triticum aestivum L.,C,Average of local and regional transport,S,Questionnaire supplemented by expert estimatio...,Czech Republic,-,...,-,-,-,-,-,91%,100%,145%,-,"Data provided by author (Jelinkova, Z., person..."
8,NaN,Moudrý Jr et al. (2013b),Wheat bread,Triticum aestivum L.,O,Average of local and regional transport,S,Questionnaire supplemented by expert estimatio...,Czech Republic,-,...,-,-,-,-,-,91%,100%,145%,-,"Data provided by author (Jelinkova, Z., person..."
9,NaN,Kulak et al. (2015),Wheat bread,Triticum aestivum L.,C,Low-input integrated crop and livestock produc...,F,Direct and semi-structured interviews; IPCC (2...,France,-,...,-,-,-,-,-,114%,100%,Incl,2.4,"Inventory from Supplementary Materials 1, reca..."
10,NaN,Kulak et al. (2015),Wheat and rye bread,Triticum aestivum L.; Secale cereale L.,C,Animal labour; bread made on farm in wood-fire...,F,Direct and semi-structured interviews; IPCC (2...,France,-,...,-,-,-,-,-,99%,100%,Incl,1.7,"Inventory from Supplementary Materials 1, reca..."
11,NaN,Kulak et al. (2015),Wheat bread,Triticum aestivum L.,C,Large-scale,M,Data provided by French Chamber of Commerce; I...,France,Beauce area,...,-,-,-,-,-,109%,100%,Incl,2.1,"Inventory from Supplementary Materials 1, reca..."


## Total land use for UK

### Multiple entries (do we actually need this? **not implemented yet**)

We also have to consider that the same product (_e.g._ coffee beans) has multiple entries (rows) in the data coming from different studies (need to avoid double counting). In theory the weights take into account this

$\rightarrow$ wait for Joseph's reply (how weights are calculated/should be used).

### Land use for animal products
Calculate:
- total land for animal product (pastoral land) $\Rightarrow$ summed over all items
- weighted average of pastoral land only (permanent + temporary) 

of European countries vs UK.

In [7]:
def get_pastural_land_and_wavg(dataset, select_country='all', fillnan=True, animal_items=[]):
    '''
    dataset : dataset - the dataset to consider (either animal only or animal + plant)
    select_country : str of list of str - 'all' considers all countries in the dataset, otherwise list of counstries to consider 
    fillnan : bool - convert nan values to zero
    animal_items : list of str - list animal items to select 
    --- 
    Returns the total pastural land (temp + perm) array per item, total land summed over all items, and its weighted average for the selected countries
    '''
    # select animal items and selected countries
    dataset[dataset['Product'].isin(animal_items)]
    if select_country != 'all':
        filtered_dataset = dataset[dataset['Country'].isin(select_country)]
    else:
        filtered_dataset = dataset
    
    w = filtered_dataset['Weight'].str.replace('%', '').astype(float) / 100
    past_land = filtered_dataset['Temp Past'].astype(float) + filtered_dataset['Perm Past'].astype(float)
    if fillnan:
        past_land = past_land.fillna(0)
    # convert past_land into array containing only the values
    past_land_arr = past_land.values
    # weighted average and sum of all land
    wavg = np.average(past_land, weights=w)
    weighted_past_land = w * past_land
    tot_past_land = weighted_past_land.sum()
    
    return past_land_arr, tot_past_land, wavg

# land used/kg --> array of land per item, tot land summed over all items, land averaged over all items
past_land_arr_EU_UK, tot_past_land_EU_UK, wavg_past_land_EU_UK = get_pastural_land_and_wavg(dataset_csv_animal, select_country='all', animal_items=animal_product_types)
past_land_arr_UK, tot_past_land_UK, wavg_past_land_UK = get_pastural_land_and_wavg(dataset_csv_animal, select_country=['United Kingdom'], animal_items=animal_product_types)
print(f"Weighted average of pasture land (EU + UK): {wavg_past_land_EU_UK} m2/kg")
print(f"Weighted average of pasture land (UK): {wavg_past_land_UK} m2/kg")


Weighted average of pasture land (EU + UK): 12.760537439613525 m2/kg
Weighted average of pasture land (UK): 14.128227194492254 m2/kg


### Land use for plant products
Calculate:
- total land for plant product (arable land) $\Rightarrow$ summed over all items
weighted average of arable land

of European countries vs UK.

**Question:** we should consider also land used for animal feed, right? _i.e._ Seed etc in land use for animal products

In [8]:
# some values in the columns 'Seed', 'Arable', 'Fallow', 'Loss' are '-' => convert them to zero
arable_land_type = ['Seed', 'Arable', 'Fallow', 'Loss']
for land in arable_land_type:
    dataset_csv_plant[land] = dataset_csv_plant[land].replace('-', '0')
    dataset_csv_animal[land] = dataset_csv_animal[land].replace('-', '0')
    dataset_csv[land] = dataset_csv[land].replace('-', '0')

/tmp/ipykernel_3414467/1242423807.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_csv_plant[land] = dataset_csv_plant[land].replace('-', '0')
/tmp/ipykernel_3414467/1242423807.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_csv_animal[land] = dataset_csv_animal[land].replace('-', '0')


In [9]:
def get_arable_land_and_wavg(dataset, select_country='all', fillnan=True):
    '''
    dataset : dataset - the dataset to consider (plant + animal)
    select_country : str of list of str - 'all' considers all countries in the dataset, otherwise list of counstries to consider 
    fillnan : bool - convert nan values to zero
    ---
    Returns the total arable land (animal + plant) array per item, total land summed over all items, and its weighted average for the selected countries
    '''
    # wavg_tot_arable_land = wavg_seed + wavg_arable + wavg_fallow + wavg_loss
    # first calculate all wavgs (same weights for all land types)
    if select_country != 'all':
        filtered_dataset = dataset[dataset['Country'].isin(select_country)]
        w = filtered_dataset['Weight'].str.replace('%', '').astype(float) / 100
    else:
        filtered_dataset = dataset
        w = dataset['Weight'].str.replace('%', '').astype(float) / 100
    
    # Sum the land columns
    arable_land = filtered_dataset[arable_land_type].astype(float).sum(axis=1)
    
    if fillnan:
        arable_land = arable_land.fillna(0)
    # convert dataset into array containing only the values
    arable_land_arr = arable_land.values
    # weighted average and sum of all land
    wavg = np.average(arable_land, weights=w)  
    weighted_aable_land = w * arable_land
    tot_arable_land = weighted_aable_land.sum()
    return arable_land_arr, tot_arable_land, wavg

# land used/kg --> array of land per item, tot land summed over all items, land averaged over all items
arable_land_arr_EU_UK, tot_arable_land_EU_UK, wavg_arable_land_EU_UK = get_arable_land_and_wavg(dataset_csv, select_country='all')
arable_land_arr_UK, tot_arable_land_UK, wavg_arable_land_UK = get_arable_land_and_wavg(dataset_csv, select_country=['United Kingdom'])

### Land and production
In order to get the land use factor per each item we need the total land for animal and one for plant product (pastoral and arable land). Therefore to first get the total land used for both in m2, we can just multiply the total land used in m2/FU by the total animal/plant production (of UK).

Columns LH onward in spreadsheet should indicate production (all kind of production, from produced gas to marketable product) -- quite unsure how to interpret all of them, expecially for plant products.
Animal production should be indicated by column PL which contains the ``Marketable final weight (kg liveweight)'' (which I don't think is the only column that has to be used) $\rightarrow$ following cell uses this value to get land used in $\rm m^2$ as: 

$\rm tot\, partoral\, land\, [m^2] =\, land\, [m^2/kg] \times production\, [kg]$

**Question:** How to do this for plant products? Is this even correct at all? I believe that the production (for animal products) is more complicated than just taking values in column PL...

In [10]:
# select ''M F Wt (kg LW)' in dataset_amimal for United Kingdom only
animal_production_UK = dataset_csv_animal[dataset_csv_animal['Country'] == 'United Kingdom']['M F Wt (kg LW)']
# replace '-' with 0 and convert to float
animal_production_UK = animal_production_UK.replace('-', '0').astype(float)
w_animal = dataset_csv_animal[dataset_csv_animal['Country'].isin(['United Kingdom'])]['Weight'].str.replace('%', '').astype(float) / 100
prod_x_land = past_land_arr_UK * animal_production_UK * w_animal

print('animal produciton', animal_production_UK.sum())
print(f'total pastoral land (sum of all item) = {prod_x_land.sum():.2f} m2  --> first land*prod per each item then summed together')
print(f"(tot past land/kg) * production = {(tot_past_land_UK * animal_production_UK.sum()):.2f} m2  --> sum first (get tot land and tot production) and then multiply the two.")

animal produciton 2102.1000000000004
total pastoral land (sum of all item) = 890.75 m2  --> first land*prod per each item then summed together
(tot past land/kg) * production = 17255.09 m2  --> sum first (get tot land and tot production) and then multiply the two.


The result is different between the two methods because some items have zero in production or land, but if you sum first you don't account for this, _i.e._ if prod=[1, 0, 3, ...] and land=[0, 2, 1, ...] you get 4*3=12 if you first sum then multiply and you get 3 if you do the opposite. 
It makes more sense to me if we first multiply, but if we want to compare with the result obtained with FAOSTAT data we need to sum first.

Here instead we calculate the same total pastoral land, but using tot animal production from FAOSTAT 

In [11]:
# values taken from matching spreadsheet (which are taken from agrifoodpy, taken from FAO); values in kg
tot_UK_mass_produciton_animal_FAOSTAT = 21736700000
tot_UK_mass_produciton_plant_FAOSTAT = 42192000000

# land_used * produciton
tot_past_land_UK_FAOSTAT = tot_past_land_UK * tot_UK_mass_produciton_animal_FAOSTAT
tot_arable_land_UK_FAOSTAT = tot_arable_land_UK * tot_UK_mass_produciton_plant_FAOSTAT
print(f'tot pastoral land (land [m2/kg] * production [kg]) for UK: {tot_past_land_UK_FAOSTAT/1e10} *10^10 m2')
print(f'tot arable land (land [m2/kg] * production [kg]) for UK: {tot_arable_land_UK_FAOSTAT/1e10} *10^10 m2')

tot pastoral land (land [m2/kg] * production [kg]) for UK: 17.842570195000004 *10^10 m2
tot arable land (land [m2/kg] * production [kg]) for UK: 92.65722882318269 *10^10 m2


Tot pastoral and arable land from calculator is or the order of $10^{10}\rm m^2$ -- expected to be different, but this is a bit too much I would say. 

## Land use factor per item (ongoing)

Calculated for each item (UK), returns a dataset.

$\rm land\, use\, factor\, (item)\, = \frac{\rm land\, used\, item}{\rm all\, land\, used\, for\, agri \, or \, past}$

In [12]:
land_use_factor_per_item = prod_x_land / (tot_past_land_UK_FAOSTAT + tot_arable_land_UK_FAOSTAT)
land_use_factor_per_item

2071    1.829650e-10
2072    3.518558e-10
2131    0.000000e+00
2132    0.000000e+00
2133    0.000000e+00
2134    0.000000e+00
2135    0.000000e+00
2174    2.712157e-10
2175    0.000000e+00
2244    0.000000e+00
2245    0.000000e+00
2246    0.000000e+00
2247    0.000000e+00
2248    7.022637e-14
2249    0.000000e+00
2250    0.000000e+00
2251    0.000000e+00
2295    0.000000e+00
2296    0.000000e+00
2297    0.000000e+00
2298    0.000000e+00
2299    0.000000e+00
2300    0.000000e+00
2301    0.000000e+00
2439    0.000000e+00
2440    0.000000e+00
2441    0.000000e+00
2442    0.000000e+00
2443    0.000000e+00
2444    0.000000e+00
2445    0.000000e+00
2604    0.000000e+00
2605    0.000000e+00
2606    0.000000e+00
2607    0.000000e+00
2608    0.000000e+00
2609    0.000000e+00
2610    0.000000e+00
2642    0.000000e+00
2643    0.000000e+00
2644    0.000000e+00
2645    0.000000e+00
2697    0.000000e+00
2698    0.000000e+00
dtype: float64

### Scale land factors
Scale all the land use factors associated with arable and animal products so the total arable and pasture land matches the totals in the UK